# 06 · "Is it a coffee bean?" (anomaly detection)

**Problem:** the classifier always picks one of the four classes. In app testing a coin was
labeled Defect at 100% confidence.

**Method (Mahalanobis distance on deep features, Lee et al., 2018):**
1. Take MobileNetV2's 1,280 *features* for every training bean (its learned description of
   the image, just before the final layer).
2. Compress them to 64 numbers with PCA.
3. Describe each class by its center and the shared spread around the centers.
4. Anomaly score of a new image = squared Mahalanobis distance to the **nearest** class center.
5. Threshold = the score that 97.5% of real validation beans stay under.

The score is built into the phone model as a second output, so the app only checks
whether `score / threshold > 1`.

**Run on Colab (GPU helps).** About 10 minutes, mostly feature extraction and downloads.

In [ ]:
# --- SETUP: run this cell first -----------------------------------------
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Change this if you uploaded the project folder somewhere else in your Drive:
    PROJECT = Path("/content/drive/MyDrive/kapeklasipika")
    %pip install -q imagehash
else:
    # Running on your laptop: walk up from this notebook until we find config/config.yaml
    PROJECT = Path.cwd()
    while not (PROJECT / "config" / "config.yaml").exists():
        PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.config import load_config, use_local_copy
cfg = load_config()
print("Project folder:", PROJECT)

In [ ]:
# Colab only: copy the images to Colab's fast local disk (Drive is slow for 8,000 files)
if IN_COLAB:
    cfg = use_local_copy(cfg)

In [ ]:
import json, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from src.data_loading import clean_split_files, load_clean_split, CLASS_NAMES
from src.ood import (build_feature_model, extract, MahalanobisDetector, synthetic_coins,
                     ood_report, build_phone_model, to_tflite_two_outputs, tflite_run_two_outputs)
from src.evaluate import show_image_grid

MODELS = cfg["paths"]["models"]; METRICS = cfg["paths"]["metrics"]; FIG = cfg["paths"]["figures"]
SIZE = cfg["img_size"]
BEST = (METRICS / "best_model.txt").read_text().strip()
print("Model:", BEST)
model = tf.keras.models.load_model(MODELS / f"{BEST}.keras")
feature_model = build_feature_model(model)

files = clean_split_files(cfg)
train_ds, val_ds, test_ds = (load_clean_split(cfg, s, files=files) for s in ("train", "val", "test"))

## 1. Extract features (train, validation, test)

In [ ]:
f_train, p_train, y_train = extract(feature_model, train_ds)
f_val, p_val, y_val = extract(feature_model, val_ds)
f_test, p_test, y_test = extract(feature_model, test_ds)
print("train", f_train.shape, "| val", f_val.shape, "| test", f_test.shape)

## 2. Fit the detector on training beans, set the threshold on validation beans

In [ ]:
detector = MahalanobisDetector(n_components=64).fit(f_train, y_train)
threshold = detector.set_threshold(f_val, keep=0.975)
pca_var = detector.pca.explained_variance_ratio_.sum()
print(f"PCA keeps {pca_var:.1%} of the feature variance with 64 components")
print(f"Threshold (97.5% of validation beans below): {threshold:.1f}")
s_val, s_test = detector.score(f_val), detector.score(f_test)

## 3. Non-bean test images
- **tf_flowers:** 500 photos of flowers (natural objects, far from beans).
- **CIFAR-10:** 500 small photos of animals and vehicles, upscaled.
- **Synthetic coins:** 300 coin-like discs on a light background, a stand-in for the coin
  we tested in the app.

None of these were used to fit the detector or choose the threshold.

In [ ]:
%pip install -q importlib_resources   # tensorflow_datasets needs it on newer Colab images
import tensorflow_datasets as tfds

def tfds_images(name, split, n):
    ds = tfds.load(name, split=split, as_supervised=True, shuffle_files=False)
    ds = ds.take(n).map(lambda x, y: (tf.image.resize(tf.cast(x, tf.float32), (SIZE, SIZE)), 0))
    return ds.batch(32)

coins = synthetic_coins(300, SIZE)
coin_ds = tf.data.Dataset.from_tensor_slices((coins, np.zeros(len(coins), int))).batch(32)
ood_feats = {
    "tf_flowers": extract(feature_model, tfds_images("tf_flowers", "train", 500))[0],
    "CIFAR-10": extract(feature_model, tfds_images("cifar10", "test", 500))[0],
    "Synthetic coins": extract(feature_model, coin_ds)[0],
}
fig, axes = plt.subplots(1, 6, figsize=(12, 2.3))
for ax, img in zip(axes, coins[:6]):
    ax.imshow(img.astype("uint8")); ax.axis("off")
fig.suptitle("Synthetic coin-like test images"); plt.tight_layout()
plt.savefig(FIG / "06_synthetic_coins.png", dpi=200); plt.show()

## 4. Results

In [ ]:
ood_scores = {k: detector.score(v) for k, v in ood_feats.items()}
report = ood_report(detector, s_test, ood_scores)
report.to_csv(METRICS / "ood_report.csv", index=False)
display(report)

plt.figure(figsize=(8, 4))
bins = np.linspace(0, np.quantile(np.r_[s_test, *ood_scores.values()], 0.99), 60)
plt.hist(s_test, bins=bins, alpha=0.6, label="Test beans", density=True)
for k, s in ood_scores.items():
    plt.hist(s, bins=bins, alpha=0.45, label=k, density=True)
plt.axvline(threshold, color="red", ls="--", label="threshold")
plt.xlabel("Anomaly score (squared Mahalanobis distance)"); plt.ylabel("Density")
plt.title("Beans vs non-bean images"); plt.legend(); plt.tight_layout()
plt.savefig(FIG / "06_anomaly_scores.png", dpi=200); plt.show()

**How to read this:** `flagged_as_not_bean` should be about 2.5% for real beans (by design)
and as close to 100% as possible for the non-bean sets. AUROC is the chance that a random
non-bean image scores higher than a random bean (1.0 = perfect separation, 0.5 = guessing).

## 5. Bonus: does a high score also mean "hard bean"?
If flagged test beans are misclassified more often, the flag doubles as an
"unclear, check this one by hand" warning.

In [ ]:
flagged = s_test > threshold
correct = p_test.argmax(1) == y_test
summary = pd.DataFrame({
    "group": ["accepted beans", "flagged beans"],
    "n": [int((~flagged).sum()), int(flagged.sum())],
    "accuracy": [correct[~flagged].mean(), correct[flagged].mean() if flagged.any() else np.nan],
})
summary.to_csv(METRICS / "ood_flagged_accuracy.csv", index=False)
display(summary)
print("Flagged test beans per class:", {c: int(flagged[y_test == i].sum()) for i, c in enumerate(CLASS_NAMES)})

## 6. Optional: your own non-bean photos
Put any photos of non-bean objects (coins, rice, corn, a pen...) in
`02_data/external_non_beans/` and run this cell.

In [ ]:
own = PROJECT / "02_data" / "external_non_beans"
paths = sorted(p for p in own.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}) if own.exists() else []
if paths:
    imgs = np.stack([tf.keras.preprocessing.image.smart_resize(
        tf.keras.utils.img_to_array(tf.keras.utils.load_img(p)), (SIZE, SIZE)) for p in paths])
    f_own = extract(feature_model, tf.data.Dataset.from_tensor_slices((imgs, np.zeros(len(imgs), int))).batch(16))[0]
    r = detector.score(f_own) / threshold
    display(pd.DataFrame({"file": [p.name for p in paths], "score/threshold": r.round(2), "not_a_bean": r > 1}))
else:
    print("No photos in 02_data/external_non_beans/ (optional).")

## 7. Export the new phone model (probabilities + anomaly ratio)
The previous single-output model is kept as `kapeklasipika_v1.tflite`.

In [ ]:
phone_model = build_phone_model(feature_model, detector)
check = np.concatenate([np.concatenate([x.numpy() for x, _ in test_ds.take(2)]), coins[:16]])

v1 = MODELS / "kapeklasipika.tflite"
if v1.exists() and not (MODELS / "kapeklasipika_v1.tflite").exists():
    shutil.copy2(v1, MODELS / "kapeklasipika_v1.tflite")

out = MODELS / "kapeklasipika.tflite"
size_mb, precision, rel_err = to_tflite_two_outputs(phone_model, out, check)
print(f"Saved {out.name}: {size_mb} MB, {precision} weights, anomaly ratio within {rel_err:.2%} of Keras")

correct = total = flagged_n = 0
for images, labels in test_ds:
    p, r = tflite_run_two_outputs(out, images.numpy())
    correct += int((p.argmax(1) == labels.numpy()).sum()); total += len(p); flagged_n += int((r > 1).sum())
print(f"TFLite test accuracy: {correct / total:.2%} | test beans flagged as not-a-bean: {flagged_n / total:.1%}")

json.dump({"model": BEST, "n_components": 64, "keep": 0.975, "threshold": threshold,
           "pca_variance_kept": float(pca_var), "tflite_mb": size_mb, "precision": precision},
          open(METRICS / "ood_params.json", "w"), indent=2)

assets = PROJECT / "07_deployment" / "mobile_app" / "assets"
assets.mkdir(parents=True, exist_ok=True)
shutil.copy2(out, assets / "kapeklasipika.tflite")
shutil.copy2(MODELS / "labels.txt", assets / "labels.txt")
print("Copied to", assets)